In [1]:
import hicstraw
import numpy as np
import math
import os

os.chdir('../..')
from load import generate_coverage_profile_intra_inter, load_GC_comtent, getmappability, getGCpercent, getFragmentsNumber, load_num_res_site, load_bin_dir_to_ratio
from binlib import output_bin
from metric import calculate_correlation_with_groundtruth, calculate_precision_racall, calculate_dict_correlation
from convert import convert_bin_to_bw

# back to current filder
os.chdir('Exps/exp10_keep_redundant_bin')

ref_genome_folder = '../../Data/ref_genome/hg19'
exclude_chr = ['chrM', 'chrY']
resolution = 50000


# load ref genome length info
chr_length = dict()
with open(ref_genome_folder + '/hg19.len', 'r') as f:
    lines = f.read().splitlines()
    for line in lines:
        chr_name , len_ = line.split('\t')[:2]
        if chr_name in exclude_chr:
            continue
        chr_length[chr_name] = int(len_)
print('chr length : \n',chr_length)

# calculate bin number in each chr
chr_bin_number = dict()
for key in chr_length.keys():
    chr_bin_number[key] = math.floor(chr_length[key] / resolution) + 1
print('chr bin number : \n', chr_bin_number)



chr length : 
 {'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276, 'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022, 'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895, 'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753, 'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520, 'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560}
chr bin number : 
 {'chr1': 4986, 'chr2': 4864, 'chr3': 3961, 'chr4': 3824, 'chr5': 3619, 'chr6': 3423, 'chr7': 3183, 'chr8': 2928, 'chr9': 2825, 'chr10': 2711, 'chr11': 2701, 'chr12': 2678, 'chr13': 2304, 'chr14': 2147, 'chr15': 2051, 'chr16': 1808, 'chr17': 1624, 'chr18': 1562, 'chr19': 1183, 'chr20': 1261, 'chr21': 963, 'chr22': 1027, 'chrX': 3106}


### Merge residual to orig file and output to binfile

In [3]:
bias_rowsum = open('bias_allchroms_nonzeros.txt','r')
residual = open('residual.txt','r')

res_dict = dict()
for key in chr_bin_number.keys():
    res_dict[key] = np.zeros(chr_bin_number[key])

bias_rowsum_lines = bias_rowsum.readlines()[1:]
residual_line = residual.readlines()

for i, bias_line in enumerate(bias_rowsum_lines):
    bias_line_split = bias_line.strip().split()
    chr_ = bias_line_split[0]
    bin_num = int(bias_line_split[1])
    res = residual_line[i]
    res_dict[chr_][bin_num] = float(res)

In [17]:
def output_bin(
    dict_ : dict, 
    trg_output_dir : str, 
    trg_name : str = '',
    keep_redundant = True) -> None:

    if not os.path.isdir(trg_output_dir):
        os.mkdir(trg_output_dir)
    config_file = open(f'{trg_output_dir}/{trg_name}_config.txt', 'w+')
    config_file.write('chr\tpath\n')

    for key in dict_.keys():
        bin_file_path = f'{trg_output_dir}/{trg_name}_{key}.bin'
        config_file.write(f'{key}\t{bin_file_path}\n')
        with open(bin_file_path, 'w+') as f:
            f.write(f'start\tend\tobs\texp\tvar\n')
            min_value =  min(dict_[key])
            for i, value in enumerate(dict_[key]):
                start_ = i * resolution
                end_ = ( i +1 ) * resolution - 1
                obs = value - min_value
                exp = 0 - min_value
                var = obs - exp
                # only store if var == 0
                if not var == 0:
                    f.write(f'{start_}\t{end_}\t{obs}\t{exp}\t{var}\n')
                elif keep_redundant:
                    f.write(f'{start_}\t{end_}\t0\t0\t0\n')
    config_file.close()

In [18]:
# output bin file for segmentation
output_bin(res_dict, trg_output_dir = 'exp10_bin')

# output for visualization
convert_bin_to_bw('exp10_bin', './exp10.bw')

### Run BICseq-seq manually 

(linux)
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp10_bin/_config.txt exp10_bin/exp10_lambda3.seg --title HiC_K562_keep_redundant_bin_lambda3 --fig exp10_bin/exp10_lambda3.png --bootstrap --lambda 3
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp10_bin/_config.txt exp10_bin/exp10_lambda2.seg --title HiC_K562_keep_redundant_bin_lambda2 --fig exp10_bin/exp10_lambda2.png --bootstrap --lambda 2
../BICseq2-seg_v0.7.3/BICseq2-seg.pl exp10_bin/_config.txt exp10_bin/exp10_lambda1.seg --title HiC_K562_keep_redundant_bin_lambda2 --fig exp10_bin/exp10_lambda1.png --bootstrap --lambda 1
 

### metric

In [19]:
lambda_ = [3,2,1]

for l in lambda_:
    print(f"lambda {l}")
    calculate_correlation_with_groundtruth(f'exp10_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    calculate_precision_racall(f'exp10_lambda{l}.seg',groundtruth_file = '../../CNV/K562_WGS_CNV/K562_WGS_CNV_lambda4.txt')
    print('\n')

lambda 3
Pearson corr. :  0.684
Spearman corr. :  0.827
Number of ground truth : 74
Number of target : 23
Number of hits : 18
precision : 0.783
recall : 0.243

hits by size : 
	 < 1Mb :  0
	1Mb - 2Mb :  0
	2Mb - 5Mb :  1
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  8
	>= 50Mb :  7
sum of  hit size : 791838020
average hit fraction : 0.835


lambda 2
Pearson corr. :  0.84
Spearman corr. :  0.84
Number of ground truth : 74
Number of target : 30
Number of hits : 24
precision : 0.8
recall : 0.324

hits by size : 
	 < 1Mb :  1
	1Mb - 2Mb :  1
	2Mb - 5Mb :  2
	5Mb - 10Mb :  1
	10Mb - 20Mb :  1
	20Mb - 50Mb :  10
	>= 50Mb :  8
sum of  hit size : 919838014
average hit fraction : 0.883


lambda 1
Pearson corr. :  0.842
Spearman corr. :  0.861
Number of ground truth : 74
Number of target : 44
Number of hits : 33
precision : 0.75
recall : 0.446

hits by size : 
	 < 1Mb :  3
	1Mb - 2Mb :  1
	2Mb - 5Mb :  4
	5Mb - 10Mb :  3
	10Mb - 20Mb :  2
	20Mb - 50Mb :  12
	>= 50Mb :  8
sum of  hit size : 